In [1]:
import sys
sys.path.append('../../datasets')
#from spark_utils import *
import getpass
import pathlib
import os
import io
from tqdm import tqdm
from optuna.samplers import TPESampler
#import datadicts as dd
import pathlib
import sys
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sklearn
import catboost as cb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn import metrics
import optuna
import shap
#from optuna.samplers import TPESampler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, precision_recall_curve, roc_curve
import warnings
warnings.filterwarnings('ignore')

import pickle

from sklearn.preprocessing import LabelEncoder

/opt/user-venvs/python3.8/lib64/python3.8/site-packages/shap/utils/_clustering.py:35: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def _pt_shuffle_rec(i, indexes, index_mask, partition_tree, M, pos):
/opt/user-venvs/python3.8/lib64/python3.8/site-packages/shap/utils/_clustering.py:54: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def delta_minimization_order(al

In [2]:
final_df = pd.read_parquet('final_new.parquet')

# Forest

In [22]:
from sklearn.ensemble import RandomForestClassifier

In [23]:
x = final_df.drop(columns=['target'])
y = final_df['target']

In [24]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 0)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size = 0.25, random_state = 0)

In [25]:
cat_features = x_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_features = x_train.select_dtypes(include=['int64', 'int32', 'float64', 'float32']).columns.tolist()

In [26]:
x_train_encoded = x_train.copy()
x_val_encoded = x_val.copy()
x_test_encoded = x_test.copy()

In [27]:
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
x_train_encoded[cat_features] = encoder.fit_transform(x_train_encoded[cat_features])
x_val_encoded[cat_features] = encoder.transform(x_val_encoded[cat_features])
x_test_encoded[cat_features] = encoder.transform(x_test_encoded[cat_features])

## First iter

In [30]:
model_forest = RandomForestClassifier(n_estimators=500, max_depth=5, random_state=0, n_jobs=-1)
model_forest.fit(x_train_encoded, y_train)

RandomForestClassifier(max_depth=5, n_estimators=500, n_jobs=-1, random_state=0)

In [31]:
print(f"TRAIN : {roc_auc_score(y_train, model_forest.predict_proba(x_train_encoded)[:, 1])}")
print(f"VAL : {roc_auc_score(y_val, model_forest.predict_proba(x_val_encoded)[:, 1])}")
print(f"TEST : {roc_auc_score(y_test, model_forest.predict_proba(x_test_encoded)[:, 1])}")

TRAIN : 0.728326249438576
VAL : 0.7067552284229693
TEST : 0.7061491836385034


In [32]:
print(f"TRAIN : {f1_score(y_train, model_forest.predict(x_train_encoded))}")
print(f"VAL : {f1_score(y_val, model_forest.predict(x_val_encoded))}")
print(f"TEST : {f1_score(y_test, model_forest.predict(x_test_encoded))}")

TRAIN : 0.0
VAL : 0.0
TEST : 0.0


In [33]:
print(classification_report(y_test, model_forest.predict(x_test_encoded)))

              precision    recall  f1-score   support

         0.0       0.67      1.00      0.80     12285
         1.0       0.00      0.00      0.00      6086

    accuracy                           0.67     18371
   macro avg       0.33      0.50      0.40     18371
weighted avg       0.45      0.67      0.54     18371



In [35]:
importance = pd.DataFrame({
    'feature': x_train_encoded.columns,
    'importance': model_forest.feature_importances_
}).sort_values(by='importance', ascending=False)

In [36]:
importance.head(30)

,feature,importance
628,srv_mb_full_qty_sum_12m,0.070187
1415,dep_mnth_payroll_min_qty_sum_3m,0.052175
1618,dep_acct_tot_qty_sum_3m,0.051212
1520,crd_dc_act_spend_qty_sum_12m,0.038626
1516,crd_tot_act_qty_sum_6m,0.035308
1524,crd_dc_open_qty_sum_9m,0.031961
627,srv_thanks_mnth_qty_sum_12m,0.030105
648,crd_otf_total_rub_amt_sum_12m,0.022478
1613,crd_tot_act_spend_qty_sum_6m,0.019929
1438,crd_cc_open_qty_sum_6m,0.018615


In [43]:
best_features = (importance.loc[importance['importance'] > 0.001, 'feature'].tolist())

In [44]:
len(best_features)

131

## Second iter

In [45]:
x_train_encoded_new = x_train_encoded[best_features]
x_val_encoded_new = x_val_encoded[best_features]
x_test_encoded_new = x_test_encoded[best_features]

In [51]:
model_forest = RandomForestClassifier(n_estimators=1000, max_depth=5, random_state=0, n_jobs=-1)
model_forest.fit(x_train_encoded_new, y_train)

RandomForestClassifier(max_depth=5, n_estimators=1000, n_jobs=-1,
                       random_state=0)

In [52]:
print(f"TRAIN : {roc_auc_score(y_train, model_forest.predict_proba(x_train_encoded_new)[:, 1])}")
print(f"VAL : {roc_auc_score(y_val, model_forest.predict_proba(x_val_encoded_new)[:, 1])}")
print(f"TEST : {roc_auc_score(y_test, model_forest.predict_proba(x_test_encoded_new)[:, 1])}")

TRAIN : 0.7355955281819473
VAL : 0.7168911422798718
TEST : 0.7161217702952832


In [53]:
print(f"TRAIN : {f1_score(y_train, model_forest.predict(x_train_encoded_new))}")
print(f"VAL : {f1_score(y_val, model_forest.predict(x_val_encoded_new))}")
print(f"TEST : {f1_score(y_test, model_forest.predict(x_test_encoded_new))}")

TRAIN : 0.0
VAL : 0.0
TEST : 0.0


In [55]:
print(pd.Series(model_forest.predict(x_train_encoded_new)).value_counts())

0.0    55112
Name: count, dtype: int64


In [56]:
print(pd.Series(model_forest.predict_proba(x_test_encoded_new)[:, 1]).describe())

count    18371.000000
mean         0.334213
std          0.164498
min          0.037594
25%          0.151875
50%          0.430865
75%          0.461760
max          0.485004
dtype: float64
